In [ ]:
print("="*70)
print("ArbItro Training Pipeline 2 - Multi-Clip Multi-View")
print("="*70)

import tensorflow as tf
print(f"\nTensorFlow version: {tf.__version__}")

gpu_name = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if gpu_name:
    gpu_info = gpu_name[0].split(',')
    print(f"  GPU  : {gpu_info[0].strip()}")
    print(f"  VRAM : {gpu_info[1].strip()}")

    print("\n  System Resources:")
    !free -h | grep Mem
    !nvidia-smi --query-gpu=memory.free,memory.total --format=csv
else:
    print("\n  No GPU found")

print("\n" + "="*70)

In [ ]:
import sys
import os

try:
    from google.colab import drive
    USE_COLAB = True
    drive.mount('/content/drive')
    BASE_DIR            = '/content/drive/MyDrive/ArbItro'
    CODE_DIR            = os.path.join(BASE_DIR, 'ArbItro_code')
    DRIVE_ROOT          = os.path.join(BASE_DIR, 'ArbItro_dataset')
    DRIVE_TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')
except ImportError:
    USE_COLAB = False

    BASE_DIR            = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
    CODE_DIR            = os.path.join(BASE_DIR, 'model', 'src', 'pipeline2')
    DRIVE_ROOT          = BASE_DIR
    DRIVE_TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')

sys.path.insert(0, CODE_DIR)

print(f"\n  USE_COLAB      : {USE_COLAB}")
print(f"  Dataset root   : {DRIVE_ROOT}")
print(f"  Training root  : {DRIVE_TRAINING_ROOT}")

required_paths = [
    os.path.join(DRIVE_ROOT, 'dataset', 'train', 'annotations.json'),
    os.path.join(DRIVE_ROOT, 'dataset', 'valid', 'annotations.json'),
    os.path.join(DRIVE_ROOT, 'dataset', 'train'),
    os.path.join(DRIVE_ROOT, 'dataset', 'valid'),
]

print("\nVerifying dataset structure:")
all_ok = True
for path in required_paths:
    exists = os.path.exists(path)
    status = "Found" if exists else "Not found"
    if not exists:
        all_ok = False
    print(f"  {status}: {path}")

In [ ]:
# Install required packages
!pip install opencv-python-headless

In [ ]:
if USE_COLAB:
    !mkdir -p /content/arbitro_model
    %cd /content/arbitro_model
    !cp {CODE_DIR}/data_loader.py .
    !cp {CODE_DIR}/model.py .
    sys.path.insert(0, '/content/arbitro_model')
    print("\nFiles copied in the working directory:")
    !ls -lh

In [ ]:
try:
    from data_loader import ArbItroDataGenerator
    from model import (
        build_arbitro_model_speed_aware_lstm_multiclip,
        compile_arbitro_model,
        BinaryBalancedAccuracy,
        MulticlassBalancedAccuracy,
    )
    print("  Imports successful!")
except Exception as e:
    print(f"  Import error: {e}")
    raise

In [ ]:
import numpy as np
import traceback

CONFIG = {
    'train_json'      : os.path.join(DRIVE_ROOT, 'dataset', 'train', 'annotations.json'),
    'train_video_dir' : os.path.join(DRIVE_ROOT, 'dataset', 'train'),
    'valid_json'      : os.path.join(DRIVE_ROOT, 'dataset', 'valid', 'annotations.json'),
    'valid_video_dir' : os.path.join(DRIVE_ROOT, 'dataset', 'valid'),
    'batch_size'      : 8,
    'max_clips'       : 4,
    'dim'             : (224, 398),
    'n_frames'        : 16,
}

try:
    train_gen = ArbItroDataGenerator(
        json_path=CONFIG['train_json'],
        base_video_path=CONFIG['train_video_dir'],
        batch_size=CONFIG['batch_size'],
        max_clips=CONFIG['max_clips'],
        dim=CONFIG['dim'],
        n_frames=CONFIG['n_frames'],
        shuffle=True,
        use_auxiliary_features=True,
        augment=True,
    )

    val_gen = ArbItroDataGenerator(
        json_path=CONFIG['valid_json'],
        base_video_path=CONFIG['valid_video_dir'],
        batch_size=CONFIG['batch_size'],
        max_clips=CONFIG['max_clips'],
        dim=CONFIG['dim'],
        n_frames=CONFIG['n_frames'],
        shuffle=False,
        use_auxiliary_features=True,
        augment=False,
    )

    print(f"  Train Samples  : {train_gen.n_samples} (Augmented)")
    print(f"  Valid Samples  : {val_gen.n_samples} (Clean/Original)")
    print(f"  Train Batches  : {len(train_gen)}")
    print(f"  Valid Batches  : {len(val_gen)}")

    # Sanity check on a single batch
    X_test, Y_test = train_gen[0]
    print(f"\n  Input Keys         : {list(X_test.keys())}")
    print(f"  video_input shape  : {X_test['video_input'].shape}")
    print(f"  speed_input shape  : {X_test['speed_input'].shape}")
    print(f"  Output Keys        : {list(Y_test.keys())}")
    print(f"  head_severity shape: {Y_test['head_severity'].shape}")

    # Class weights
    training_class_weights = train_gen.get_class_weights()
    print(f"\n  Severity Weights : {training_class_weights['head_severity']}")
    print(f"  Offence Weights  : {training_class_weights['head_offence']}")
    print(f"  Action Weights   : {training_class_weights['head_action']}")

except Exception as e:
    print(f"  Error: {e}")
    traceback.print_exc()
    raise

In [ ]:
print(f"  Architecture : InceptionResNetV2 + BiLSTM")
print(f"  Input shape  : ({CONFIG['n_frames']}, {CONFIG['dim'][0]}, {CONFIG['dim'][1]}, 3)")
print(f"  Max clips    : {CONFIG['max_clips']}")

model = build_arbitro_model_speed_aware_lstm_multiclip(
    frame_input_shape=(CONFIG['n_frames'], *CONFIG['dim'], 3),
    max_clips=CONFIG['max_clips'],
)
print("  Model built!")

model = compile_arbitro_model(model)

print(f"\n  Model Statistics:")
print(f"  Total parameters : {model.count_params():,}")

trainable_params     = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = model.count_params() - trainable_params
print(f"  Trainable        : {trainable_params:,}")
print(f"  Non-trainable    : {non_trainable_params:,}")

print(f"\n  GPU Memory After Model Build:")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
print("="*70)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau,
    TensorBoard, CSVLogger, Callback,
)
import datetime

CHECKPOINT_DIR = os.path.join(DRIVE_TRAINING_ROOT, 'checkpoints')
LOGS_DIR = os.path.join(DRIVE_TRAINING_ROOT, 'logs')
BACKUP_DIR = os.path.join(DRIVE_TRAINING_ROOT, 'backups')

for directory in [DRIVE_TRAINING_ROOT, CHECKPOINT_DIR, LOGS_DIR, BACKUP_DIR]:
    os.makedirs(directory, exist_ok=True)

run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "_OFFENCE_FOCUS"
tensorboard_log_dir = os.path.join(LOGS_DIR, run_id)


class SaveEveryNEpochs(Callback):
    def __init__(self, save_dir, n_epochs=2):
        super().__init__()
        self.save_dir = save_dir
        self.n_epochs = n_epochs

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.n_epochs == 0:
            val_off = logs.get('val_head_offence_bal_acc', 0.0)
            filepath = os.path.join(
                self.save_dir,
                f'backup_ep{epoch + 1:02d}_Off{val_off:.2f}.keras'
            )
            try:
                self.model.save(filepath)
                print(f"\n  [Backup] Saved ep{epoch + 1}")
            except:
                pass


TARGET_METRIC = 'val_head_offence_bal_acc'
TARGET_LOSS = 'val_head_offence_loss'

callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, 'BEST_OFFENCE_FOCUSED.keras'),
        monitor=TARGET_METRIC,
        save_best_only=True,
        mode='max',
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor=TARGET_LOSS,
        factor=0.5,
        patience=6,
        mode='min',
        min_lr=1e-7,
        verbose=1,
        cooldown=1,
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        mode='min',
        restore_best_weights=True,
        verbose=1,
    ),
    TensorBoard(log_dir=tensorboard_log_dir, histogram_freq=1),
    CSVLogger(os.path.join(DRIVE_TRAINING_ROOT, 'history_offence_focus.csv'), append=True),
    SaveEveryNEpochs(BACKUP_DIR, n_epochs=2),
]

print(f"  Callbacks configured!")
print(f"  Target metric  : {TARGET_METRIC}")
print(f"  LR monitor     : {TARGET_LOSS}")
print(f"  TensorBoard    : {tensorboard_log_dir}")

if USE_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir {LOGS_DIR}

In [ ]:
from datetime import datetime

INITIAL_EPOCH = 0
model.optimizer.learning_rate.assign(1e-4)

start_time = datetime.now()
print("\n" + "=" * 70)
print("  TRAINING STARTED")
print("=" * 70)
print(f"\n  Start time : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Steps      : {len(train_gen)} train / {len(val_gen)} val")
print(f"  Epochs     : {INITIAL_EPOCH} / 50")
print("=" * 70 + "\n")

try:
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=50,
        initial_epoch=INITIAL_EPOCH,
        callbacks=callbacks,
        verbose=1,
    )
except KeyboardInterrupt:
    print("\n" + "=" * 70)
    print("  Training Interrupted")
    print("=" * 70)

end_time = datetime.now()
training_duration = end_time - start_time
print(f"\n  Training duration: {training_duration}")

In [ ]:
import json

MODELS_DIR = os.path.join(DRIVE_TRAINING_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

final_path = os.path.join(DRIVE_TRAINING_ROOT, 'models', 'pipeline2.keras')
os.makedirs(os.path.dirname(final_path), exist_ok=True)
model.save(final_path)
print(f"Model saved: {final_path}")

history_path = os.path.join(LOGS_DIR, 'history_offence_focus.json')
history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(history_path, 'w') as f:
    json.dump(history_dict, f, indent=2)
print(f"  History saved: {history_path}")
print(f"  Duration     : {training_duration}")